# Study B granular invariance and controllability analysis

Keep this notebook separate from the canonical `study_b_analysis.ipynb`.

Use it for:

- persona-level and risk-level invariance slices
- pressure-style / pressure-schedule breakdowns
- controllability intensity curves
- exemplar bad-vs-good flips and early-turn instability

In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import pandas as pd

RUNTIME_ROOT = Path.cwd().resolve().parents[0]
CASE_DELTAS_PATH = RUNTIME_ROOT / "metric-results" / "invariance_smoke" / "qwen3-lmstudio" / "study_b_case_deltas.json"
CONTROLLABILITY_PATH = RUNTIME_ROOT / "metric-results" / "controllability" / "qwen3-lmstudio" / "study_b_archived_smoke.json"

case_rows = json.loads(CASE_DELTAS_PATH.read_text(encoding="utf-8")) if CASE_DELTAS_PATH.exists() else []
case_df = pd.DataFrame(case_rows)
control_payload = json.loads(CONTROLLABILITY_PATH.read_text(encoding="utf-8")) if CONTROLLABILITY_PATH.exists() else {}

print(f"case rows: {len(case_df)}")
display(case_df.head() if not case_df.empty else pd.DataFrame())
control_payload.get("variants", [])[:1]

In [ ]:
if not case_df.empty:
    if "persona" not in case_df.columns and "metadata" in case_df.columns:
        case_df["persona"] = case_df["metadata"].apply(lambda value: value.get("persona_id") if isinstance(value, dict) else None)
    if "risk" not in case_df.columns and "strata" in case_df.columns:
        case_df["risk"] = case_df["strata"].apply(lambda value: value.get("risk") if isinstance(value, dict) else None)

    display(
        case_df.groupby(["metric", "persona"], as_index=False)
        .agg(mean_delta=("delta", "mean"), n=("id", "count"))
        .sort_values(["metric", "mean_delta"], ascending=[True, False])
        .head(40)
    )

records = []
for variant in control_payload.get("variants", []):
    for metric_name, metric in variant.get("metrics", {}).items():
        records.append({
            "tag": variant.get("tag"),
            "variant_type": variant.get("variant_type"),
            "intensity": variant.get("intensity"),
            "metric": metric_name,
            "delta_c": metric.get("delta_c"),
            "ci_low": metric.get("ci_low"),
            "ci_high": metric.get("ci_high"),
        })
control_df = pd.DataFrame(records)
display(control_df)

if not control_df.empty:
    for metric_name in sorted(control_df["metric"].unique()):
        subset = control_df[control_df["metric"] == metric_name].sort_values(["intensity", "tag"])
        plt.figure(figsize=(8, 4))
        plt.plot(subset["intensity"], subset["delta_c"], marker="o")
        plt.title(f"Study B controllability curve: {metric_name}")
        plt.xlabel("intensity")
        plt.ylabel("delta_C")
        plt.tight_layout()
        plt.show()